In [103]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "projects" else Path.cwd()
DIR = PROJECT_ROOT / "data"
ACTS_DIR = PROJECT_ROOT / "artifacts"
M_STATE = 42

np.random.seed(M_STATE)

In [104]:
import tarfile
import urllib.request

def load_used_cars(data_path: str):
  return pd.read_csv(data_path / "used_cars" / "used_cars.csv")

used_cars_full = load_used_cars(PROJECT_ROOT / "datasets")

In [106]:
used_cars_full.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [132]:
# digging into mileage
# kmpl is diesel, petrol. km/kg is CNG, LPG
used_cars_full.loc[used_cars_full["mileage"].apply(lambda row: isinstance(row, str) and "kmpl" not in row), ["mileage", "fuel"]]
#used_cars_full["mileage"].str.extract(r'([\d.]+)')

,mileage,fuel
6,17.3 km/kg,LPG
35,33.44 km/kg,CNG
90,26.2 km/kg,LPG
203,26.6 km/kg,CNG
402,20.88 km/kg,CNG
...,...,...
7308,26.83 km/kg,CNG
7543,33.44 km/kg,CNG
7642,13.2 km/kg,LPG
7733,21.94 km/kg,CNG


In [139]:
# digging into torque
# torque contains values with and without rpm. rpm is either rpm or RPM.
# there are values that are missing rpm information
# proposal. remove rows with missing rpm information. split the torque value into 2 features: torque and rpm.
# different values for torque: 28.3@ 1,700-2,200(kgm@ rpm), 110(11.2)@ 4800
# for torque, normalize it into Nm from KGM, nm, 110(11.2). 
# For RPM, we can create a new feature, it's the second numerical value in the string. Option 1 is to create categories based on ranges. Option 2 is to have single values, where if a range is presented, we take the lower bound or higher bound. 
used_cars_full.loc[used_cars_full["torque"].apply(lambda row: isinstance(row, str)), ["torque"]].sample(25)

pattern = used_cars_full["torque"].dropna().str.replace(r'[\d,]*\.?\d+', '#', regex=True)
pattern.value_counts()

torque
#Nm@ #rpm            4669
#Nm@ #-#rpm          2269
#@ #(kgm@ rpm)        260
#nm@ #-#rpm           133
#@ #-#(kgm@ rpm)      131
#nm@ #rpm             114
# Nm at #-#rpm         51
# kgm at #-#rpm        41
#Nm                    34
# kgm at # rpm         31
#kgm@ #rpm             28
#Nm at # rpm           22
# Nm at #-# rpm        22
#Nm at #-# rpm         17
# Nm at #  rpm         13
#Nm@ #+/-#rpm          12
# KGM at #-# RPM       10
# Nm at # rpm           7
# / #                   7
#NM@ #-#rpm             5
#Nm@ #-#                4
# KGM at # RPM          3
# Nm at # rpm           3
#NM@ #rpm               3
#Nm@ #                  3
#Nm@ #~#rpm             2
#@ #-#rpm               2
#@ #+/-#(NM@ rpm)       2
#@ #-#                  2
# Nm at #rpm            2
#Nm(#kgm)@ #rpm         1
#(#)@ #                 1
# Nm /# rpm             1
#  Nm at #  rpm         1
Name: count, dtype: int64

In [135]:
# Digging into horse power
used_cars_full.loc[used_cars_full["max_power"].apply(lambda row: isinstance(row, str) and "bhp" not in row), ["max_power"]]

,max_power
575,0
576,0
1442,0
1443,0
2549,0
2550,0


In [138]:
used_cars_full["km_driven"].max()

np.int64(2360457)

In [155]:
# digging into car make
# test assumption that within a car make and ignoring the model, there is not too much price range

used_cars_full[["name", "selling_price"]].assign(make=lambda df: df["name"].apply(lambda row: row.split(" ")[0])).groupby("make").agg({"selling_price": ["min", "max", "mean", "std"]})

selling_price                                      
                        min       max          mean           std
make                                                             
Ambassador            75000    200000  1.240000e+05  5.417872e+04
Ashok                300000    300000  3.000000e+05           NaN
Audi                 730000   6523000  2.612200e+06  1.388758e+06
BMW                  480000   7200000  4.109917e+06  1.841158e+06
Chevrolet             45000   1295000  2.738675e+05  1.745688e+05
Daewoo                45000    100000  7.700000e+04  2.858321e+04
Datsun               150000    590000  3.146000e+05  8.909525e+04
Fiat                  70000    700000  2.960638e+05  1.464282e+05
Force                555000   1050000  8.875000e+05  1.820920e+05
Ford                  55000   3250000  5.166826e+05  3.887911e+05
Honda                 50000   2125000  5.961780e+05  2.907487e+05
Hyundai               30000   1950000  4.585544e+05  2.644723e+05
Isuzu               1160000   2700000  1.942000e+06  7.485453e+05
Jaguar              2000000   3500000  2.915465e+06  3.195644e+05
Jeep                1325000   4100000  2.149613e+06  9.912411e+05
Kia                  643000   1900000  1.504500e+06  5.944165e+05
Land                1650000   4500000  3.608333e+06  1.385791e+06
Lexus               5150000   5150000  5.150000e+06  0.000000e+00
MG                  1700000   1900000  1.783333e+06  1.040833e+05
Mahindra             120000   1800000  6.232241e+05  3.195101e+05
Maruti                29999   1200000  4.030757e+05  2.316082e+05
Mercedes-Benz        400000   6000000  2.470444e+06  1.509375e+06
Mitsubishi           150000   1950000  8.175000e+05  6.780508e+05
Nissan               110000   1200000  4.654074e+05  2.241022e+05
Opel                  68000     68000  6.800000e+04           NaN
Peugeot               57000     57000  5.700000e+04           NaN
Renault              200000   1265000  4.626184e+05  2.100356e+05
Skoda                100000   3251000  6.077238e+05  4.972067e+05
Tata                  40000   1700000  3.574331e+05  2.841098e+05
Toyota               100000   3500000  9.599467e+05  6.548619e+05
Volkswagen           210000   2150000  4.988172e+05  2.465477e+05
Volvo               1200000  10000000  3.272015e+06  1.158945e+06

Strategies
- Torque: Divide into 2 features: Torque (Nm/number) and RPM (rpm/number). First value before parenthesis is the Torque. For RPM ranges, grab the mean. For values in kgm, convert by multiplying by 9.80665. Impute missing values with the mean of RPM.
- Mileage: Take the numerical value and leave the fuel type alone. Let the model figure out the kmpl vs km/kg and its relationship with fuel type. 0.0 means bad data. Put it as NaN and impute.
- Max Power: Strip the string value, and take the first number. For 0 and NaN numbers, impute with the mean. If the value is only bhp, convert to NaN and impute.
- km_driven: Fine. There is a 2 million value. Let's remove that row. Empty rows are imputed.
- owner: remove rows where owner is a test drive car. not representative of the real selling market. one hot encoding.
- seller type: one hot encoding
- transmission: one hot encoding
- name: preserve the first 2 tokens of the name. apply one hot encoding. make sure that the handle_unknown is properly set

Transformer rules for Torque:

1. Determine the unit and extract the Nm value (check in this order):

- If the string contains  "Nm"  (any case:  Nm ,  nm ,  NM ) anywhere → take the first number in the string as the Nm value directly. No conversion - even if the string also contains  "kgm"  elsewhere (e.g.  "380Nm(38.7kgm)@2500rpm"  →  380  Nm, discard the  38.7 ).
- Else if the string contains  "kgm"  (any case, with or without parentheses) → take the first number in the string, multiply by 9.80665 to get Nm.
- Else (no unit word at all, e.g.  "210 / 1900" ,  "110(11.2)@ 4800" ) → take the first number as Nm directly (verified: magnitude fits the Nm range, not kgm).

2. Extract RPM - relative to the delimiter, not "last number in the string":

- Locate the first delimiter:  @ ,  at , or  / . Everything after it (up to any trailing parenthetical unit label) is the "RPM segment."
- Within that segment:
- Single number → use directly.
- Range ( #-#  or  #~# ) → mean of the first two numbers in the segment.
-  +/-  tolerance ( #+/-# ) → first number in the segment (the center); discard the tolerance.
- No number present in the segment →  NaN .
- Strip commas from numbers before parsing ( "1,900"  →  1900 ).

3. Missing RPM:

- Leave as  NaN ; impute later with  SimpleImputer(strategy="mean") , fit per-fold inside the pipeline (never on the full dataset upfront).

4. Keep torque and RPM as two separate numeric output columns; never drop a row just because RPM is missing - torque itself is still valid and usable.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class TorqueTransformer(BaseEstimator, TransformerMixin):
  def __init__(self):
    pass

  def fit(self, X, y=None):
    pass

  def transform(self, X, y=None):
    pass

In [ ]:
# used_cars_full = used_cars_full.drop_duplicates()